In [66]:
import datetime as dt
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
from securities_load.securities.postgresql_database_functions import sqlalchemy_engine
from securities_load.securities.securities_table_functions import retrieve_ohlcv_from_to

In [67]:
def localData(exchange, ticker, years):
    load_dotenv()
    engine = sqlalchemy_engine()
    exchange_code = "XNAS"
    ticker = "MSFT"
    num_of_years = 10
    start_date = dt.datetime.now() - dt.timedelta(int(365.25 * num_of_years))
    start_date_str = dt.datetime.strftime(start_date, "%Y-%m-%d")
    end_date = dt.datetime.now()
    end_date_str = dt.datetime.strftime(end_date, "%Y-%m-%d")

    df = retrieve_ohlcv_from_to(
        engine,
        exchange_code=exchange_code,
        ticker=ticker,
        start_date=start_date_str,
        end_date=end_date_str,
    )

    print(df.tail(3))

    return df

In [68]:
df = localData(exchange="XNAS", ticker="MSFT", years=1)

Exchange_id is: 2
                  date    open      high      low   close    volume
Datetime                                                           
2026-05-01  2026-05-01  412.80  417.1100  410.435  414.44  31372362
2026-05-04  2026-05-04  411.54  420.7800  410.800  413.62  28066525
2026-05-05  2026-05-05  415.32  416.7799  408.800  411.38  25700891


In [69]:
def calculate_daily_volatility(ticker_data: pd.DataFrame, window_size: int) -> pd.DataFrame:
    """
    Calculate the daily historical volatility (sigma) for a stock.

    :param ticker_data: DataFrame containing the stock data
    :param window_size: The window size for calculating rolling volatility
    :return: DataFrame with daily volatility added
    """

    # Calculate daily log returns
    ticker_data['log_return'] = np.log(ticker_data['close'] / ticker_data['close'].shift(1))
    # ticker_data.dropna(inplace=True)

    # Daily volatility (standard deviation of returns)
    ticker_data['rolling_daily_sigma'] = ticker_data['log_return'].rolling(window=window_size).std()

    # Optional: Annualize sigma (assuming 252 trading days/year)
    ticker_data['AnnualizedSigma'] = ticker_data['rolling_daily_sigma'] * np.sqrt(252)

    return ticker_data

In [70]:
df = calculate_daily_volatility(df, window_size=20)
print(df.head(25))
print(df.tail(3))

                  date     open      high       low   close    volume  \
Datetime                                                                
2018-10-26  2018-10-26  105.690  108.7500  104.7600  106.96  55523104   
2018-10-29  2018-10-29  108.105  108.7000  101.6300  103.85  55162001   
2018-10-30  2018-10-30  103.660  104.3800  100.1100  103.73  65350878   
2018-10-31  2018-10-31  105.435  108.1400  105.3900  106.81  51062383   
2018-11-01  2018-11-01  107.050  107.3200  105.5300  105.92  33384201   
2018-11-02  2018-11-02  106.480  107.3200  104.9750  106.16  37680194   
2018-11-05  2018-11-05  106.370  107.7400  105.9000  107.51  27922144   
2018-11-06  2018-11-06  107.380  108.8400  106.2800  107.72  24340248   
2018-11-07  2018-11-07  109.440  112.2400  109.4000  111.96  37901704   
2018-11-08  2018-11-08  111.800  112.2100  110.9100  111.75  25644105   
2018-11-09  2018-11-09  110.850  111.4500  108.7600  109.57  32039223   
2018-11-12  2018-11-12  109.420  109.9600  106.1000

In [88]:
df.loc['2026-05-01' : '2026-05-05']

,date,open,high,low,close,volume,log_return,rolling_daily_sigma,AnnualizedSigma
Datetime,,,,,,,,,
2026-05-01,2026-05-01,412.80,417.1100,410.435,414.44,31372362,0.016200,0.021521,0.341642
2026-05-04,2026-05-04,411.54,420.7800,410.800,413.62,28066525,-0.001981,0.021529,0.341757
2026-05-05,2026-05-05,415.32,416.7799,408.800,411.38,25700891,-0.005430,0.021609,0.343038


In [72]:
def option_delta(S, K, T, r, sigma, option_type="call"):
    """
    Calculate the Delta of a European option using the Black-Scholes model.

    Parameters:
        S (float): Current stock price
        K (float): Strike price
        T (float): Time to maturity in years
        r (float): Risk-free interest rate (as decimal, e.g., 0.05 for 5%)
        sigma (float): Volatility of the underlying asset (as decimal)
        option_type (str): "call" or "put"

    Returns:
        float: Delta value
    """
    # Input validation
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T, and sigma must be positive numbers.")
    if option_type not in ("call", "put"):
        raise ValueError("option_type must be 'call' or 'put'.")

    # Calculate d1
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))

    # Calculate Delta
    if option_type == "call":
        delta = norm.cdf(d1)  # N(d1)
    else:  # put option
        delta = norm.cdf(d1) - 1

    return delta

In [74]:
def calculate_time_to_maturity(expiry_date, current_date=dt.datetime.now().strftime("%Y-%m-%d")):
    """
    Calculate the time to maturity for an option.

    Parameters:
        current_date (str): Current date (YYYY-MM-DD)
        expiry_date (str): Option expiry date (YYYY-MM-DD)

    Returns:
        float: Time to maturity in years
    """
    current_date = dt.datetime.strptime(current_date, "%Y-%m-%d")
    expiry_date = dt.datetime.strptime(expiry_date, "%Y-%m-%d")

    T = (expiry_date - current_date).days / 365.0
    print(T)
    if T <= 0:
        raise ValueError("Option has already expired.")
    return T

In [75]:
S = 411.79 # Stock price
K = 420     # Strike price
r = 0.042   # current risk-free rate
sigma = 0.34 # 20% volatility    
expiry = "2026-09-18" # Option expiry date (YYYY-MM-DD)
current_date = dt.datetime.now().strftime("%Y-%m-%d") # Current date (YYYY-MM-DD)

T = calculate_time_to_maturity(expiry, current_date)

call_delta = option_delta(S, K, T, r, sigma, "call")
put_delta = option_delta(S, K, T, r, sigma, "put")

print(f"Call Delta: {call_delta:.4f}")
print(f"Put Delta:  {put_delta:.4f}")

0.3698630136986301
Call Delta: 0.5331
Put Delta:  -0.4669


In [76]:
from vollib.black_scholes_merton import black_scholes_merton # Option price
from vollib.black_scholes_merton.implied_volatility import implied_volatility as bsm_implied_volatility
from vollib.black_scholes_merton.greeks.analytical import delta as bsm_analytical_delta
from vollib.black_scholes_merton.greeks.numerical import delta as bsm_numerical_delta
from vollib.black_scholes_merton.greeks.analytical import gamma as bsm_analytical_gamma
from vollib.black_scholes_merton.greeks.numerical import gamma as bsm_numerical_gamma
from vollib.black_scholes_merton.greeks.analytical import rho as bsm_analytical_rho
from vollib.black_scholes_merton.greeks.numerical import rho as bsm_numerical_rho
from vollib.black_scholes_merton.greeks.analytical import theta as bsm_analytical_theta
from vollib.black_scholes_merton.greeks.numerical import theta as bsm_numerical_theta
from vollib.black_scholes_merton.greeks.analytical import vega as bsm_analytical_vega
from vollib.black_scholes_merton.greeks.numerical import vega as bsm_numerical_vega


In [92]:
S = 411.79     # Stock price
K = 420     # Strike price
r = 0.042   # current risk-free rate
sigma = 0.12 # 20% volatility    
t = T # Time to maturity in years
q = 0.0088 # Dividend yield (as decimal, e.g., 0.02 for 2%)
flag = 'c'
call_bsm_analytical_delta_calc = bsm_analytical_delta(flag, S, K, t, r, sigma, q)
call_bsm_analytical_gamma_calc = bsm_analytical_gamma(flag, S, K, t, r, sigma, q)
call_bsm_analytical_rho_calc = bsm_analytical_rho(flag, S, K, t, r, sigma, q)
call_bsm_analytical_theta_calc = bsm_analytical_theta(flag, S, K, t, r, sigma, q)
call_bsm_analytical_vega_calc = bsm_analytical_vega(flag, S, K, t, r, sigma, q)
call_bsm_price_calc = black_scholes_merton(flag, S, K, t, r, sigma, q)   
call_bsm_implied_vol_calc = bsm_implied_volatility(call_bsm_price_calc, S, K, t, r, q, flag)    
flag = 'p'
put_bsm_analytical_delta_calc = bsm_analytical_delta(flag, S, K, t, r, sigma, q)
put_bsm_analytical_gamma_calc = bsm_analytical_gamma(flag, S, K, t, r, sigma, q)
put_bsm_analytical_rho_calc = bsm_analytical_rho(flag, S, K, t, r, sigma, q)
put_bsm_analytical_theta_calc = bsm_analytical_theta(flag, S, K, t, r, sigma, q)
put_bsm_analytical_vega_calc = bsm_analytical_vega(flag, S, K, t, r, sigma, q)

print(f"Call Delta (BSM Analytical): {call_bsm_analytical_delta_calc:.4f}")
print(f"Put Delta (BSM Analytical):  {put_bsm_analytical_delta_calc:.4f}")
print(f"Call Gamma (BSM Analytical): {call_bsm_analytical_gamma_calc:.4f}")
print(f"Put Gamma (BSM Analytical):  {put_bsm_analytical_gamma_calc:.4f}")
print(f"Call Rho (BSM Analytical): {call_bsm_analytical_rho_calc:.4f}")
print(f"Put Rho (BSM Analytical):  {put_bsm_analytical_rho_calc:.4f}")
print(f"Call Theta (BSM Analytical): {call_bsm_analytical_theta_calc:.4f}")
print(f"Put Theta (BSM Analytical):  {put_bsm_analytical_theta_calc:.4f}")
print(f"Call Vega (BSM Analytical): {call_bsm_analytical_vega_calc:.4f}")
print(f"Put Vega (BSM Analytical):  {put_bsm_analytical_vega_calc:.4f}")
print(f"Call Price (BSM Analytical): {call_bsm_price_calc:.4f}")
print(f"Call Implied Volatility (BSM Analytical): {call_bsm_implied_vol_calc:.4f}")

call_bsm_numerical_delta_calc = bsm_numerical_delta('c', S, K, t, r, sigma, q)
put_bsm_numerical_delta_calc = bsm_numerical_delta('p', S, K, t, r, sigma, q)
print(f"Call Delta (BSM Numerical): {call_bsm_numerical_delta_calc:.4f}")
print(f"Put Delta (BSM Numerical):  {put_bsm_numerical_delta_calc:.4f}")

Call Delta (BSM Analytical): 0.4722
Put Delta (BSM Analytical):  -0.5245
Call Gamma (BSM Analytical): 0.0132
Put Gamma (BSM Analytical):  0.0132
Call Rho (BSM Analytical): 0.6804
Put Rho (BSM Analytical):  -0.8491
Call Theta (BSM Analytical): -0.0606
Put Theta (BSM Analytical):  -0.0230
Call Vega (BSM Analytical): 0.9937
Put Vega (BSM Analytical):  0.9937
Call Price (BSM Analytical): 10.5178
Call Implied Volatility (BSM Analytical): 0.1200
Call Delta (BSM Numerical): 0.4722
Put Delta (BSM Numerical):  -0.5245


In [38]:
from vollib.black_scholes import black_scholes # Option price
from vollib.black_scholes.implied_volatility import implied_volatility as bs_implied_volatility
from vollib.black_scholes.greeks.analytical import delta as bs_analytical_delta
from vollib.black_scholes.greeks.numerical import delta as bs_numerical_delta

In [87]:
S = 411.89     # Stock price
K = 420     # Strike price
r = 0.042   # current risk-free rate
sigma = 0.19 # 20% volatility    
t = T # Time to maturity in years
q = 0.0088 # Dividend yield (as decimal, e.g., 0.02 for 2%)
flag = 'c'
call_analytical_delta_calc = bs_analytical_delta(flag, S, K, t, r, sigma)
flag = 'p'
put_analytical_delta_calc = bs_analytical_delta(flag, S, K, t, r, sigma)

print(f"Call Delta (BS Analytical): {call_analytical_delta_calc:.4f}")
print(f"Put Delta (BS Analytical):  {put_analytical_delta_calc:.4f}") 

call_numerical_delta_calc = bs_numerical_delta('c', S, K, t, r, sigma)
put_numerical_delta_calc = bs_numerical_delta('p', S, K, t, r, sigma)
print(f"Call Delta (BS Numerical): {call_numerical_delta_calc:.4f}")
print(f"Put Delta (BS Numerical):  {put_numerical_delta_calc:.4f}")

Call Delta (BS Analytical): 0.5094
Put Delta (BS Analytical):  -0.4906
Call Delta (BS Numerical): 0.5093
Put Delta (BS Numerical):  -0.4907
